==== VQCC v5 FRONTIER BLEND  |  ONE PASTE, ONE CELL ======================
This cell GENERATES /kaggle/working/submission.zip - the archive that was
uploaded to the competition.  It does not describe it; it rebuilds it.
#
1. Add Input -> Datasets -> manwithacat/vqcc-apache-public-circuits
2. Paste this whole block into ONE code cell.
3. Run All.   It takes ~1-3 minutes (the dqi circuit alone is 530 MB, so
packing the archive is the slow step).
4. Save Version -> open the version's Output tab -> download
submission.zip.  THAT file is the submission.  Nothing else is.
#
Every one of the 18 members is pinned by SHA-256 and every gate fails
closed: a missing file, an unpinned member or a hash mismatch stops the
cell before anything is packaged.
#
THE THREE CIRCUITS ARE OTHER PEOPLE'S APACHE-2.0 PUBLIC WORK.  This cell
assembles and pins them; it does not build them.  The competition's own
verifier receipts for these exact bytes are in the public repo:
github.com/EigenOS-Quantum/eigenos-vqcc
#
  invsqrt    53000387a86b   62 qubits   Leo Tang, VQCC T7421 Submission
  dqi        748b01a7a076  891 qubits   alex chilton, 24.51 LB portfolio
  dfthcsos   37204439f717  428 qubits   Heliosli, "The dataset is public now"
#
Measured scores (the competition's own verifiers, default salt):
  invsqrt      98,061.19483958285   -> factor 21.178349
  dqi   375,374,174,093.81177       -> factor 65.106939
  dfthcsos 312,444,127.2287831      -> factor 10.739839
  geometric mean 24.556835   (our previous public score was 24.43253)
#
The competition data itself is NOT redistributed by this notebook - the
members are fetched from the public Apache-2.0 dataset on Kaggle, and the
rules forbid redistributing competition data to non-participants.
---------------------------------------------------------------------------
These two exist so this exact cell can be proved byte-for-byte outside
Kaggle.  On Kaggle you never touch them.
Leave this False.  The competition's own verifiers are a BONUS check, not
the gate.  The fusion verifier is a full-domain run over 3,133,452 values
(about 1-2 hours on a 2-CPU Kaggle session) and the dqi verifier needs
~4 GB of RAM.  The gate is the SHA-256 pin set below.
---------------------------------------------------------------------------


In [ ]:
import os, sys, json, glob, hashlib, shutil, zipfile, tempfile, subprocess

DATASET = "manwithacat/vqcc-apache-public-circuits"

INPUT_ROOT = os.environ.get("VQCC_INPUT_ROOT", "/kaggle/input")
WORK = os.environ.get("VQCC_WORK_DIR", "/kaggle/working")

RUN_OFFICIAL_VERIFIERS = False

LAYOUT = [
    [
        "invsqrt.kmb",
        "circuits/invsqrt/53000387a86b.kmb"
    ],
    [
        "dqi.kmb",
        "circuits/dqi/748b01a7a076.kmb"
    ],
    [
        "dfthcsos/submission.npz",
        "factorizations/37204439f717.npz"
    ],
    [
        "dfthcsos/angle_k_0.kmb",
        "circuits/chemistry/37204439f717/angle_k_0.kmb"
    ],
    [
        "dfthcsos/angle_k_1.kmb",
        "circuits/chemistry/37204439f717/angle_k_1.kmb"
    ],
    [
        "dfthcsos/angle_k_2.kmb",
        "circuits/chemistry/37204439f717/angle_k_2.kmb"
    ],
    [
        "dfthcsos/angle_k_3.kmb",
        "circuits/chemistry/37204439f717/angle_k_3.kmb"
    ],
    [
        "dfthcsos/angle_k_4.kmb",
        "circuits/chemistry/37204439f717/angle_k_4.kmb"
    ],
    [
        "dfthcsos/angle_k_5.kmb",
        "circuits/chemistry/37204439f717/angle_k_5.kmb"
    ],
    [
        "dfthcsos/angle_k_6.kmb",
        "circuits/chemistry/37204439f717/angle_k_6.kmb"
    ],
    [
        "dfthcsos/angle_k_7.kmb",
        "circuits/chemistry/37204439f717/angle_k_7.kmb"
    ],
    [
        "dfthcsos/angle_k_8.kmb",
        "circuits/chemistry/37204439f717/angle_k_8.kmb"
    ],
    [
        "dfthcsos/angle_k_9.kmb",
        "circuits/chemistry/37204439f717/angle_k_9.kmb"
    ],
    [
        "dfthcsos/angle_k_10.kmb",
        "circuits/chemistry/37204439f717/angle_k_10.kmb"
    ],
    [
        "dfthcsos/angle_k_11.kmb",
        "circuits/chemistry/37204439f717/angle_k_11.kmb"
    ],
    [
        "dfthcsos/angle_k_12.kmb",
        "circuits/chemistry/37204439f717/angle_k_12.kmb"
    ],
    [
        "dfthcsos/angle_k_13.kmb",
        "circuits/chemistry/37204439f717/angle_k_13.kmb"
    ],
    [
        "dfthcsos/angle_k_14.kmb",
        "circuits/chemistry/37204439f717/angle_k_14.kmb"
    ]
]
EXPECTED = {
    "invsqrt.kmb": "53000387a86b2cfa495c68944ecc7fbe8f80d14801b2aa9b99f6d01b412dfd5e",
    "dqi.kmb": "748b01a7a07662c3f7e389b844e313a3d55dfb3b3d993414e5ad3dfac6fe154c",
    "dfthcsos/submission.npz": "65ec5b26ed58ff7f52793de21c8cca95c84a06ba33279732f934bdcdf5178036",
    "dfthcsos/angle_k_0.kmb": "956dc0b494a85b72aacc830b45e10e52a65a11454207fb76072b43c050016f8e",
    "dfthcsos/angle_k_1.kmb": "cf0291d59ba246eb2bbf61f56b7128a69bdca36d740f2bf8487fe4b0fa77c88c",
    "dfthcsos/angle_k_2.kmb": "7d2038cea60c2a9484d51e4ea60521e254dd4c3e2f29133ed62a101a3d088557",
    "dfthcsos/angle_k_3.kmb": "9937a233c65981c29aa923f2821819dac54faac17a55d1dfc06353bff09ba681",
    "dfthcsos/angle_k_4.kmb": "063aa910bb8e3f3cd4864762ea4accecc4f043ddf6ab8d55dc3461a858af88fb",
    "dfthcsos/angle_k_5.kmb": "b381c166926b4196ab26e1a428bf8d530ce18af86453a5c9cb8824a8f0980c99",
    "dfthcsos/angle_k_6.kmb": "6dd19bf59ca80deacf851f42eeee3d785d8334346642bb94e22c99904060010c",
    "dfthcsos/angle_k_7.kmb": "2375754fa132843532bf075b56bf9a4d3ab877cedbb8cc22777a59867cd850f8",
    "dfthcsos/angle_k_8.kmb": "5708aea25df768f3c58daa759af70c5e4cadeb38b7e3680d67e93312cba5607e",
    "dfthcsos/angle_k_9.kmb": "f1f1cc1cbe2ad7d74a425888bdee3780956c2ba555b12e5f00b2aaf71b6c7cc1",
    "dfthcsos/angle_k_10.kmb": "8c31a64784c3d0ecb6bf71c5e67a4f27bdf63ce0f4dd4e213ff4ee20029d9ba9",
    "dfthcsos/angle_k_11.kmb": "402ebdf216abae95c91ae2aee37078a2bbb13ef59bf0db66f4af9f057161e310",
    "dfthcsos/angle_k_12.kmb": "7707330d587dfc5609e3f2b7ebca3dfacf64afb77d462050457671e80bcc168c",
    "dfthcsos/angle_k_13.kmb": "451cc5acbe1bbefd0affef3622a7e611eb7d3cd784c6d26a59d7312354f91c39",
    "dfthcsos/angle_k_14.kmb": "c36fef91c85080a9b686c8c81d995c0569fead3f79960673d7d26ccce9dd3217"
}
VERIFIED_ARCHIVE_SHA256 = "a25c038c923ceca62ba373810715ed8b2d8796444e76143a2d959f168e3d886e"
VERIFIED_ARCHIVE_BYTES = 26271346

def find_dataset_root():
    slug = DATASET.split("/")[-1]
    cands = []
    for base in (INPUT_ROOT, "input", "."):
        cands += [os.path.join(base, "datasets", DATASET),
                  os.path.join(base, DATASET),
                  os.path.join(base, slug)]
    for c in cands:
        if os.path.isdir(c) and os.path.exists(os.path.join(c, "circuits")) \
           and os.path.isdir(os.path.join(c, "factorizations")):
            return c
    for base in (INPUT_ROOT, "input", "."):
        for hit in sorted(glob.glob(os.path.join(base, "**",
                                                 "circuits/invsqrt/53000387a86b.kmb"),
                                 recursive=True)):
            root = hit.split("/circuits/")[0]
            if os.path.exists(os.path.join(root, "factorizations",
                                           "37204439f717.npz")):
                return root
    return None


DS = find_dataset_root()
print("dataset root :", DS)
print("python       :", sys.version.split()[0])
if DS is None:
    raise SystemExit("STOP: attach the dataset '" + DATASET +
                     "' (Add Input -> Datasets) and re-run this cell.")

CIRCUITS = os.path.join(WORK, "vqcc", "circuits")
if os.path.isdir(CIRCUITS):
    shutil.rmtree(CIRCUITS)
os.makedirs(CIRCUITS)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


rows, bad = [], 0
for member, remote in LAYOUT:
    src = os.path.join(DS, remote)
    dst = os.path.join(CIRCUITS, member.replace("/", os.sep))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(src):
        print("MISSING IN DATASET", remote)
        bad += 1
        continue
    shutil.copyfile(src, dst)
    got = sha256(dst)
    want = EXPECTED.get(member)
    if want is None:
        bad += 1
        os.remove(dst)
        print("%-5s %-30s no expected SHA-256 is pinned" % ("FAIL", member))
        continue
    ok = (got == want)
    if not ok:
        bad += 1
        os.remove(dst)
    rows.append({"member": member, "sha256": got,
                 "bytes": os.path.getsize(dst) if os.path.exists(dst) else 0,
                 "ok": ok, "expected": want, "upstream": remote})
    print("%-5s %-30s %s" % ("OK" if ok else "FAIL", member, got))

print()
print("%d of %d members OK, %d bad" % (len(rows) - bad, len(rows), bad))
if bad:
    raise SystemExit("STOP: a hash did not match. Nothing was packaged.")

OUT = os.path.join(WORK, "submission.zip")
FIXED = (2026, 11, 4, 0, 0, 0)

if os.path.exists(OUT):
    os.remove(OUT)
print("packing (the dqi circuit is 530 MB, so this is the slow step)...")
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as z:
    for member, _remote in LAYOUT:
        path = os.path.join(CIRCUITS, member.replace("/", os.sep))
        info = zipfile.ZipInfo(member, date_time=FIXED)
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o644 << 16
        info.create_system = 3
        with open(path, "rb") as fh:
            z.writestr(info, fh.read(), compresslevel=9)

stage = tempfile.mkdtemp(prefix="vqcc_pack_")
with zipfile.ZipFile(OUT) as z:
    names = z.namelist()
    assert len(names) == 18, "expected 18 members, got %d" % len(names)
    z.extractall(stage)

manifest = {"members": []}
for member, _remote in LAYOUT:
    p = os.path.join(stage, member.replace("/", os.sep))
    assert os.path.exists(p), "member missing from the packed archive: " + member
    digest = sha256(p)
    want = EXPECTED[member]
    assert digest == want, "packed member does not match its pin: " + member
    manifest["members"].append({"member": member, "sha256": digest,
                                 "bytes": os.path.getsize(p)})
shutil.rmtree(stage, ignore_errors=True)

manifest["archive_sha256"] = sha256(OUT)
manifest["archive_bytes"] = os.path.getsize(OUT)
os.makedirs(os.path.join(WORK, "receipts"), exist_ok=True)
with open(os.path.join(WORK, "receipts", "submission-manifest.json"), "w") as fh:
    json.dump(manifest, fh, indent=2)

print("archive        :", OUT)
print("archive bytes  :", manifest["archive_bytes"])
print("archive sha256 :", manifest["archive_sha256"])
print("members        :", len(manifest["members"]))
if manifest["archive_sha256"] == VERIFIED_ARCHIVE_SHA256:
    print("archive check  : MATCH - byte-identical to the verified archive")
else:
    print("archive check  : the 18 MEMBERS are pinned and identical; only the")
    print("                 zip compression bytes differ between zlib builds.")
    print("                 Upload this file.")

if RUN_OFFICIAL_VERIFIERS:
    try:
        VERIFIERS = None
        for root in (INPUT_ROOT, "input", "."):
            for hit in sorted(glob.glob(os.path.join(root, "**", "verifiers"),
                                        recursive=True)):
                if all(os.path.isdir(os.path.join(hit, p))
                       for p in ("invsqrt", "dqi", "dfthcsos")):
                    VERIFIERS = hit
                    break
            if VERIFIERS:
                break
        if not VERIFIERS:
            print("verifiers      : not found - skipped")
        else:
            COMP_ROOT = os.path.dirname(os.path.normpath(VERIFIERS))
            env = dict(os.environ)
            env["PYTHONPATH"] = os.pathsep.join(
                [os.path.join(COMP_ROOT, "kickmix")] +
                [p for p in env.get("PYTHONPATH", "").split(os.pathsep)
                 if p and os.path.normpath(p) != os.path.normpath(COMP_ROOT)])
            for pkg, script, args in (
                    ("invsqrt", "verify_submission_invsqrt.py",
                     [os.path.join(CIRCUITS, "invsqrt.kmb")]),
                    ("dqi", "verify_submission_dqi.py",
                     [os.path.join(CIRCUITS, "dqi.kmb")]),
                    ("dfthcsos", "verify.py", [CIRCUITS])):
                path = os.path.join(VERIFIERS, pkg, script)
                if not os.path.exists(path):
                    print("%-10s : verifier not found - skipped" % pkg)
                    continue
                print("=== %s ===" % pkg)
                proc = subprocess.run([sys.executable, path] + args,
                                      capture_output=True, text=True, env=env)
                try:
                    rep = json.loads(proc.stdout)
                    print("   status=%s score=%s" % (rep.get("status"),
                                                     rep.get("score")))
                    with open(os.path.join(WORK, "receipts",
                                           "%s-verify.json" % pkg), "w") as fh:
                        json.dump(rep, fh, indent=2, default=str)
                except Exception:
                    print("   did not return JSON:")
                    print(proc.stdout[-800:])
                    print(proc.stderr[-800:])
    except Exception as exc:
        print("optional verifier step failed (it never affects the archive):",
              exc)
else:
    print("official verifiers: SKIPPED (RUN_OFFICIAL_VERIFIERS = False)")

print()
print("UPLOAD THIS FILE: submission.zip")
print("Do not upload anything else.  Do not upload this notebook.")
